# Hybrid retrieval + optional cross-encoder reranking — a diagnostic client of `engineering_rag`

**This notebook contains no retrieval, fusion, or reranking implementation.**
It imports the production package (`engineering_rag.pipelines.retrieval_pipeline`,
`engineering_rag.services.retriever`, `engineering_rag.services.reranker`) exactly
as `engrag-retrieve` does, and demonstrates the four supported modes against
the same real, already-indexed `engineering_documents_v1` collection used by
`notebooks/03_retrieval_evaluation_demo.ipynb`.

Modes demonstrated:

1. **vector** — the existing, unchanged vector-only baseline.
2. **hybrid** — vector + BM25 + Reciprocal Rank Fusion.
3. **hybrid-rerank** — hybrid + cross-encoder reranking (`BAAI/bge-reranker-base`).

A technical-identifier query (`IEC 61511`) is also shown, since it is the
case BM25 is specifically expected to help with (see
`docs/retrieval/HYBRID_RETRIEVAL_ARCHITECTURE.md`).

This notebook is not run with a network-dependent model during CI — it is
executed locally, and its saved outputs are committed as evidence.

In [1]:
import os
import time

from engineering_rag.utils.paths import repo_root

ROOT = repo_root()
os.chdir(ROOT)  # so relative paths in YAML profiles resolve against the repo root, matching the CLI
print("Working directory:", ROOT)

Working directory: E:\engineering-rag-parser


## 1. Load the retrieval profile

In [2]:
from engineering_rag.pipelines.retrieval_config import load_retrieval_config

config = load_retrieval_config("configs/retrieval_production.yaml")
print("Embedding model:", config.embedding.model_name)
print("Collection:", config.chroma.collection_name)
print("BM25 index path:", config.bm25.index_path)
print("Reranker model:", config.reranker.model_name, "revision:", config.reranker.model_revision)

Embedding model: BAAI/bge-base-en-v1.5
Collection: engineering_documents_v1
BM25 index path: data/output/databases/bm25/engineering_documents_v1
Reranker model: BAAI/bge-reranker-base revision: 2cfc18c9415c912f9d8155881c133215df768a70


## 2. Confirm the BM25 index exists and matches the live collection

Never built implicitly by a search — this notebook assumes
`engrag-retrieve build-bm25 --profile configs/retrieval_production.yaml` was
already run (see `docs/retrieval/COMMANDS.md`).

In [3]:
from engineering_rag.databases.bm25.index import load_bm25_index

bm25_index = load_bm25_index(config.bm25)
print("BM25 corpus_count:", bm25_index.manifest.corpus_count)
print("BM25 corpus_fingerprint:", bm25_index.manifest.corpus_fingerprint[:16])
print("Document ids:", bm25_index.manifest.document_ids)
print("Source filenames:", bm25_index.manifest.source_filenames)

BM25 corpus_count: 122
BM25 corpus_fingerprint: daa15ba96d152652
Document ids: ['01e4d6fa3a2e884f83fad507d08b228aa40f814dc0f3c44e6c9db315f73c3b1a', '57f84fd5b7d2e03db2fd6bbdb3877c7f3ed3201e28a1ef7b137e4445b733432a']
Source filenames: ['Instrumentation-and-Control-Engineering.pdf', 'scanned_docling_test_image_only.pdf']


## 3. One query, three modes — provenance and ranking side by side

In [4]:
from engineering_rag.pipelines.retrieval_pipeline import run_hybrid_search

QUERY = "What is an instrument index?"

modes = {
    "vector": {"bm25_enabled": False, "reranker_enabled": False},
    "hybrid": {"bm25_enabled": True, "reranker_enabled": False},
    "hybrid-rerank": {"bm25_enabled": True, "reranker_enabled": True},
}

responses = {}
for name, kwargs in modes.items():
    started = time.perf_counter()
    responses[name] = run_hybrid_search(QUERY, config, top_k=5, **kwargs)
    elapsed = time.perf_counter() - started
    print(f"{name:15s} mode={responses[name].retrieval_mode:15s} total={elapsed * 1000:8.1f} ms")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

vector          mode=vector          total= 14074.2 ms


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

hybrid          mode=hybrid          total=  3973.0 ms


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

hybrid-rerank   mode=hybrid-rerank   total= 11068.6 ms


In [5]:
for name, response in responses.items():
    print(f"\n=== {name} ===")
    for hit in response.hits:
        print(
            f"  rank={hit.final_rank}  chunk_id={hit.chunk_id}  "
            f"vector_rank={hit.vector_rank}  bm25_rank={hit.bm25_rank}  "
            f"rrf_rank={hit.rrf_rank}  reranker_rank={hit.reranker_rank}  "
            f"reranker_score={hit.reranker_score}"
        )
        print(f"    source={hit.source_filename}  pages={hit.page_numbers}  section={hit.section_title!r}")
        print(f"    snippet: {hit.retrieval_text[:100].strip()}...")


=== vector ===
  rank=1  chunk_id=chunk_95b81ee200d7e603  vector_rank=1  bm25_rank=None  rrf_rank=None  reranker_rank=None  reranker_score=None
    source=Instrumentation-and-Control-Engineering.pdf  pages=[15]  section='3.2 Instrument Index (The Master Database)'
    snippet: Section 3: Categorized C&I Deliverables: Foundation Documents > 3.2 Instrument Index (The Master Dat...
  rank=2  chunk_id=chunk_3b7ae6d712325002  vector_rank=2  bm25_rank=None  rrf_rank=None  reranker_rank=None  reranker_score=None
    source=Instrumentation-and-Control-Engineering.pdf  pages=[15]  section='Key Content and Role'
    snippet: Section 3: Categorized C&I Deliverables: Foundation Documents > 3.2 Instrument Index (The Master Dat...
  rank=3  chunk_id=chunk_67703f828dce6e88  vector_rank=3  bm25_rank=None  rrf_rank=None  reranker_rank=None  reranker_score=None
    source=Instrumentation-and-Control-Engineering.pdf  pages=[1]  section='Instrument Index (The Master Database)'
    snippet: Section 3: Cat

## 4. Ranking comparison and latency

Notice how BM25/RRF and reranking can reorder the same candidate pool —
neither the vector-only nor the hybrid ranking is asserted here to be
"correct"; `docs/retrieval/EVALUATION.md` reports the actual measured
metrics across the full 20-case ground-truth benchmark, not this one
illustrative query.

In [6]:
for name, response in responses.items():
    print(
        f"{name:15s} candidate_counts={response.candidate_counts}  "
        f"stage_latencies_s={ {k: round(v, 4) for k, v in response.stage_latencies_s.items()} }"
    )

vector          candidate_counts={'vector': 5}  stage_latencies_s={'vector': 0.1328}
hybrid          candidate_counts={'vector': 30, 'bm25': 30, 'fused': 45}  stage_latencies_s={'vector': 0.0809, 'bm25': 0.0009, 'fusion': 0.0004}
hybrid-rerank   candidate_counts={'vector': 30, 'bm25': 30, 'fused': 45, 'reranked': 20}  stage_latencies_s={'vector': 0.0928, 'bm25': 0.0012, 'fusion': 0.0005, 'reranker': 3.2886}


## 5. A technical-identifier query — where BM25 is expected to help

`IEC 61511` is an exact standard-plus-number identifier. Compare the vector-only
ranking against the hybrid (BM25-fused) ranking for the same query.

In [7]:
IDENTIFIER_QUERY = "Find information related to IEC 61511."

vector_id_response = run_hybrid_search(IDENTIFIER_QUERY, config, top_k=5, bm25_enabled=False)
hybrid_id_response = run_hybrid_search(IDENTIFIER_QUERY, config, top_k=5, bm25_enabled=True)

print(
    "vector-only top result:", vector_id_response.hits[0].chunk_id, vector_id_response.hits[0].section_title
)
print(
    "hybrid top result:     ", hybrid_id_response.hits[0].chunk_id, hybrid_id_response.hits[0].section_title
)
print()
print("vector ranks:", [h.chunk_id for h in vector_id_response.hits])
print("hybrid ranks:", [h.chunk_id for h in hybrid_id_response.hits])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

vector-only top result: chunk_8d8e031d2a9545d4 Conclusions and Synthesis
hybrid top result:      chunk_8d8e031d2a9545d4 Conclusions and Synthesis

vector ranks: ['chunk_8d8e031d2a9545d4', 'chunk_2fb8618912ad708d', 'chunk_c83ef802ba4c475c', 'chunk_6c5a01565948bd24', 'chunk_95133c9536668ec9']
hybrid ranks: ['chunk_8d8e031d2a9545d4', 'chunk_6c5a01565948bd24', 'chunk_c83ef802ba4c475c', 'chunk_2fb8618912ad708d', 'chunk_95133c9536668ec9']


## 6. Known limitations (repeated honestly — see `docs/retrieval/EVALUATION.md`)

- BM25 improves lexical/identifier matching but has no notion of meaning.
- RRF combines rankings by position; it does not itself judge relevance.
- Cross-encoder reranking is materially slower than vector/BM25 on CPU.
- `reranker_score` is a raw model output, not a calibrated probability.
- Every ground-truth label remains `human_review_status: "machine_candidate"`.
- On this milestone's actual 20-case benchmark, vector-only retrieval had the
  best overall MRR — hybrid and reranking did **not** measurably improve
  results on this specific dataset (see the full comparison table in
  `docs/retrieval/EVALUATION.md`). This notebook demonstrates the mechanism
  working correctly, not a proven accuracy win.